In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
.appName("Cleaning Dirty Store Transactions wiht Spark Df and write PostgreSQL") \
.master("local[2]") \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 20:14:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [42]:
! curl -o /opt/examples/datasets/dirty_store_transactions.csv \
https://raw.githubusercontent.com/erkansirin78/datasets/master/dirty_store_transactions.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2548k  100 2548k    0     0  1582k      0  0:00:01  0:00:01 --:--:-- 1582k


In [4]:
df = spark.read.format("csv") \
.option("sep", ",") \
.option("header", True) \
.option("inferSchema", True) \
.load("file:///opt/examples/datasets/dirty_store_transactions.csv")

In [5]:
df.printSchema()

root
 |-- STORE_ID: string (nullable = true)
 |-- STORE_LOCATION: string (nullable = true)
 |-- PRODUCT_CATEGORY: string (nullable = true)
 |-- PRODUCT_ID: string (nullable = true)
 |-- MRP: string (nullable = true)
 |-- CP: string (nullable = true)
 |-- DISCOUNT: string (nullable = true)
 |-- SP: string (nullable = true)
 |-- Date: date (nullable = true)



In [8]:
'''
I chose to clean and cast the columns after reading the CSV because the raw dataset contains invalid characters
and currency symbols that prevent direct conversion to the target data types.
'''
from pyspark.sql import functions as F

# 1. Clean STORE_LOCATION
df2 = df.withColumn(
    "STORE_LOCATION",
    F.regexp_replace(
        F.col("STORE_LOCATION"),
        r"[^A-Za-z0-9]+$",
        ""
    )
)

# 2. Clean PRODUCT_ID and convert to INT
df2 = df2.withColumn(
    "PRODUCT_ID",
    F.regexp_replace(
        F.col("PRODUCT_ID"),
        r"[^0-9]+$",
        ""
    ).cast("int")
)

# 3. Clean monetary columns and convert to FLOAT
money_columns = ["MRP", "CP", "DISCOUNT", "SP"]

for column in money_columns:
    df2 = df2.withColumn(
        column,
        F.regexp_replace(
            F.col(column),
            r"^\$",
            ""
        ).cast("float")
    )

# 4. Convert Date to DateType
df2 = df2.withColumn(
    "Date_Casted",
    F.to_date(F.col("Date"), "yyyy-MM-dd")
)

# 5. Drop the original Date column
df2 = df2.drop("Date")


In [9]:
df2.limit(10).toPandas()

,STORE_ID,STORE_LOCATION,PRODUCT_CATEGORY,PRODUCT_ID,MRP,CP,DISCOUNT,SP,Date_Casted
0,YR7220,New York,Electronics,12254943,31.0,20.770000,1.86,29.139999,2019-11-26
1,YR7220,New York,Furniture,72619323,15.0,9.750000,1.50,13.500000,2019-11-26
2,YR7220,New York,Electronics,34161682,88.0,62.480000,4.40,83.599998,2019-11-26
3,YR7220,New York,Kitchen,79411621,91.0,58.240002,3.64,87.360001,2019-11-26
4,YR7220,New York,Fashion,39520263,85.0,51.000000,2.55,82.449997,2019-11-26
5,YR7220,New York,Kitchen,93809204,37.0,24.049999,0.74,36.259998,2019-11-26
6,YR7220,New York,Cosmetics,86610412,80.0,48.799999,6.40,73.599998,2019-11-26
7,YR7220,New York,Kitchen,52503356,71.0,42.599998,5.68,65.320000,2019-11-26
8,YR7220,New York,Kitchen,77516479,92.0,56.119999,3.68,88.320000,2019-11-26
9,YR7220,New York,Cosmetics,47334289,16.0,10.720000,0.96,15.040000,2019-11-26


In [10]:
df2.printSchema()

root
 |-- STORE_ID: string (nullable = true)
 |-- STORE_LOCATION: string (nullable = true)
 |-- PRODUCT_CATEGORY: string (nullable = true)
 |-- PRODUCT_ID: integer (nullable = true)
 |-- MRP: float (nullable = true)
 |-- CP: float (nullable = true)
 |-- DISCOUNT: float (nullable = true)
 |-- SP: float (nullable = true)
 |-- Date_Casted: date (nullable = true)



In [11]:
df2 = df2.withColumn(
    "margin_band",
    F.when(F.col("MRP") - F.col("CP") < 10, "low")
     .when(
         ((F.col("MRP") - F.col("CP")) >= 10) &
         ((F.col("MRP") - F.col("CP")) <= 30),
         "mid"
     )
     .otherwise("high")
)

In [12]:
q1_df = df2.groupBy("STORE_LOCATION").agg(
    F.sum("SP").alias("total_sp"),
    F.count("*").alias("transaction_count")
).orderBy(F.desc("total_sp"))

q1_df.limit(10).toPandas()

,STORE_LOCATION,total_sp,transaction_count
0,Washington,450798.119825,8604
1,New York,415423.889816,8064
2,Houston,388271.799817,7385
3,Miami,371295.359499,7254
4,Denver,340802.519720,6546


In [13]:
q2_df = df2.groupBy(
    "PRODUCT_CATEGORY",
    "margin_band"
).agg(
    F.avg("DISCOUNT").alias("avg_discount")
).orderBy("PRODUCT_CATEGORY", "margin_band")

q2_df.limit(10).toPandas()

,PRODUCT_CATEGORY,margin_band,avg_discount
0,Cosmetics,high,5.639794
1,Cosmetics,low,1.169634
2,Cosmetics,mid,3.663273
3,Education,high,5.490116
4,Education,low,1.270542
5,Education,mid,3.710816
6,Electronics,high,5.895865
7,Electronics,low,1.269135
8,Electronics,mid,3.947202
9,Fashion,high,5.062161


In [14]:
df2.createOrReplaceTempView("clean_transactions")
q1_sql = spark.sql("""
    SELECT
        STORE_LOCATION,
        SUM(SP) AS total_sp,
        COUNT(*) AS transaction_count
    FROM clean_transactions
    GROUP BY STORE_LOCATION
    ORDER BY total_sp DESC
""")

q1_sql.limit(10).toPandas()

,STORE_LOCATION,total_sp,transaction_count
0,Washington,450798.119825,8604
1,New York,415423.889816,8064
2,Houston,388271.799817,7385
3,Miami,371295.359499,7254
4,Denver,340802.519720,6546


In [15]:
q2_sql = spark.sql("""
    SELECT
        PRODUCT_CATEGORY,
        margin_band,
        AVG(DISCOUNT) AS avg_discount
    FROM clean_transactions
    GROUP BY PRODUCT_CATEGORY, margin_band
    ORDER BY PRODUCT_CATEGORY, margin_band
""")

q2_sql.limit(10).toPandas()

,PRODUCT_CATEGORY,margin_band,avg_discount
0,Cosmetics,high,5.639794
1,Cosmetics,low,1.169634
2,Cosmetics,mid,3.663273
3,Education,high,5.490116
4,Education,low,1.270542
5,Education,mid,3.710816
6,Electronics,high,5.895865
7,Electronics,low,1.269135
8,Electronics,mid,3.947202
9,Fashion,high,5.062161


In [16]:

#writing DB
! ls -l $SPARK_HOME/jars | grep postgresql

-rw-r--r-- 1 root      root   1221004 Oct 23 18:00 postgresql-42.7.14.jar


In [17]:
jdbc_url = "jdbc:postgresql://postgresql:5432/traindb"

df2.write \
    .format("jdbc") \
    .mode("overwrite") \
    .option("url", jdbc_url) \
    .option("dbtable", "public.clean_transactions") \
    .option("user", "train") \
    .option("password", "Ankara06") \
    .option("driver", "org.postgresql.Driver") \
    .save()

In [18]:
df2.count()

37853